In [11]:
!. ./set_paths.sh

In [26]:
import mlflow
import subprocess
import json
import os
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
import numpy as np
import re

# MLflow 실험 설정
experiment_name = "SCCL_Clustering_Experiment"
mlflow.set_experiment(experiment_name)

# 하이퍼파라미터 탐색 공간 정의 (베이지안 최적화용)
search_space = {
    "pre_train_epoch": hp.uniformint("pre_train_epoch", 0, 20),
    "joint_train_epoch": hp.uniformint("joint_train_epoch", 0, 20),
    "n_clusters": hp.uniformint("n_clusters", 5, 40),
    "batchsize": hp.choice("batchsize", [10, 20, 40]),
    "lr": hp.loguniform("lr", np.log(1e-7), np.log(1e-3)),
    "eta": hp.choice("eta", [1, 10, 100, 1000])
}

# 데이터셋 및 결과 파일 경로 설정
dataset_file = "./dstc12-data/AppenBanking/all.jsonl"
base_result_file = "./results/appen_banking_predicted"

# 기본 파라미터 설정
base_params = {
    "device": "mps",
    "model_name": "sentence-transformers/all-mpnet-base-v2",
    "temperature": 0.5,
    "alpha": 1.0
}

# 목적 함수 정의 (하이퍼파라미터 최적화를 위한)
def objective(params):
    # 기본 파라미터에 변경된 파라미터 적용
    run_params = base_params.copy()
    run_params.update(params)
    
    # 결과 파일 경로 생성 (실험별 고유 파일명)
    result_file = f"{base_result_file}_n{params['n_clusters']}_lr{params['lr']:.6f}_e{params['joint_train_epoch']}.jsonl"
    
    # MLflow 실행 시작
    with mlflow.start_run(nested=True):
        # 파라미터 로깅
        for key, value in run_params.items():
            mlflow.log_param(key, value)
        
        # PYTHONPATH 환경 변수 설정
        current_dir = os.getcwd()
        os.environ["PYTHONPATH"] = f"{os.environ.get('PYTHONPATH', '')}:{current_dir}/src/:{current_dir}/scripts/"
        
        cmd = [
            "python3", "sccl/cluster.py",
            "--device", run_params["device"],
            "--model-name", run_params["model_name"],
            "--dropout", "0.1",
            "--dataset-file", dataset_file,
            "--result-file", result_file,
            "--max-length", "100",
            "--batch-size", str(params["batchsize"]),
            "--lr", str(params["lr"]),
            "--pre-train-epoch", str(params["pre_train_epoch"]),
            "--joint-train-epoch", str(params["joint_train_epoch"]),
            "--lr-scale", "10000",
            "--augtype", "simcse",
            "--temperature", str(run_params["temperature"]),
            "--eta", str(params["eta"]),
            "--n-clusters", str(params["n_clusters"]),
            "--alpha", str(run_params["alpha"]),
            "--n-init", "100",
            "--kmeans-interval", "1",
            "--print-freq", "1",
            "--eval-interval", "1"
        ]
        
        # 실행 명령어 출력
        print(f"실행 명령어: {' '.join(cmd)}")
        
        # 명령어 실행
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        stdout, stderr = process.communicate()
        
        # 출력 로깅
        mlflow.log_text(stdout, "stdout.txt")
        mlflow.log_text(stderr, "stderr.txt")
        
        # 결과 파일이 생성되었는지 확인하고 지표 추출
        metric_value = 1000000  # 기본값
        if os.path.exists(result_file):
            mlflow.log_artifact(result_file)
            
            # stdout에서 성능 지표 추출 (print 출력 결과에서 추출)
            try:
                # stdout에서 acc, nmi 등의 지표 찾기
                metrics_found = {}
                lines = stdout.split('\n')
                # 가장 최근 출력부터 역순으로 검색
                for line in reversed(lines):
                    # 'acc: 0.123' 형태의 출력 찾기
                    for metric in ['acc', 'nmi', 'rouge_1', 'rouge_2', 'rouge_l', 'cosine_similarity']:
                        pattern = f'{metric}: ([0-9]+(?:\.[0-9]+)?)'
                        match = re.search(pattern, line)
                        if match and metric not in metrics_found:
                            value = float(match.group(1))
                            metrics_found[metric] = value
                            mlflow.log_metric(metric, value)
                
                # acc 값이 있으면 metric_value 업데이트
                if 'acc' in metrics_found:
                    metric_value = metrics_found['acc']
                    print(f"stdout에서 추출한 acc 값: {metric_value}")
                
                # 다른 지표들도 출력
                for metric, value in metrics_found.items():
                    if metric != 'acc':
                        print(f"stdout에서 추출한 {metric} 값: {value}")
            except Exception as e:
                print(f"stdout에서 지표를 추출할 수 없습니다: {e}")
        
        # 최대화할 지표의 음수 값을 반환 (hyperopt는 최소화 문제를 해결)
        print(f"metric_value: {metric_value}")
        return {
            'loss': metric_value,  # 음수 값으로 변환하여 최대화 문제를 최소화 문제로 변환
            'status': STATUS_OK,
            'result_file': result_file
        }

print("TPE 최적화를 통한 하이퍼파라미터 탐색을 시작합니다...")

with mlflow.start_run(run_name="Bayesian_Optimization"):
    trials = Trials()
    best = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,  # 베이지안 최적화 알고리즘
        max_evals=20,      # 최대 실험 횟수
        trials=trials
    )
    
    # 최적의 하이퍼파라미터 출력
    print("최적의 하이퍼파라미터:")
    print(best)
    
    # 최적의 하이퍼파라미터 로깅
    mlflow.log_params(best)
    
    # 최적의 실험 결과 로깅
    best_trial_idx = np.argmin([t['result']['loss'] for t in trials.trials])
    best_metric = -trials.trials[best_trial_idx]['result']['loss']
    mlflow.log_metric("best_metric", best_metric)
    
    print(f"최적의 성능 지표: {best_metric}")


2025/04/07 07:54:28 INFO mlflow.tracking.fluent: Experiment with name 'SCCL_Clustering_Experiment' does not exist. Creating a new experiment.


TypeError: require string label